# The Rest of the Iceberg — exploring *Titanic Birth Records* with Quarb

Pandas is the perfect tool for table-shaped data. But a table
presupposes that somebody made the data table-shaped first. This
notebook does that below-the-waterline work on **Titanic Birth Records**: the
classic `titanic.csv` plus `titanic-birth-records.json`, a fictional but
historically careful reconstruction of birth records for the ship's
passengers, drawn from eight registration systems of 1912 — eight
different tree shapes in one file.

It accompanies the article
[*The rest of the iceberg*](https://quarb.org/articles/the-rest-of-the-iceberg.html).
Setup:

```
pip install "quarb[jupyter]"
```

(on a distribution that manages its system Python, inside a
virtual environment, with Jupyter launched from it) and put `titanic.csv` and `titanic-birth-records.json` (from the
[Kaggle dataset](https://www.kaggle.com/datasets/deyanovich/titanic-birth-records) *Titanic Birth Records* or
[data.quarb.org](https://data.quarb.org/articles/titanic-birth-records-fixtures.tar.zst)) next to this
notebook. (Running inside Kaggle with the dataset attached, the
files live under `/kaggle/input/titanic-birth-records/` —
use that prefix in the `%quarb_mount` line.)

In [1]:
%load_ext quarb.ipython
%quarb_mount titanic.csv titanic-birth-records.json

mounted: titanic+titanic-birth-records (children: /titanic, /titanic-birth-records)


## First contact

The JSON is a stranger. Ask it what it is — `/*` walks children,
`:name` asks a node its name, `//` searches at any depth, and the
two pipes split the work: `|` streams one result at a time, `@|`
hands the whole stream to an aggregate.

In [2]:
%%quarb
/titanic-birth-records/archives/*:name

gro
irish_parishes
kyrkobocker
actes_de_naissance
us_state_records
ottoman_nufus
matricula_baptizatorum
metricheskie_knigi

In [3]:
%%quarb
/titanic-birth-records//passenger_id:: @| count

1146

In [4]:
%%quarb
/titanic-birth-records//*:name @| unique @| count

534

1,146 records; 534 distinct key names across eight archive
shapes. Which bureaucracy knew about how many passengers? Walk to
each archive, file its name in the register with a push `.a(...)`,
keep walking, group by the pushed name:

In [5]:
%%quarb
/titanic-birth-records/archives/* | .a(:name) | //passenger_id @| group($.a) | count | .n | %.

{'a': 'gro', 'n': 422}
{'a': 'irish_parishes', 'n': 124}
{'a': 'kyrkobocker', 'n': 166}
{'a': 'actes_de_naissance', 'n': 27}
{'a': 'us_state_records', 'n': 286}
{'a': 'ottoman_nufus', 'n': 80}
{'a': 'matricula_baptizatorum', 'n': 34}
{'a': 'metricheskie_knigi', 'n': 7}

## The table meets the tree

The famous half: 891 rows, and the famous hole — 177 rows with no
`Age`, the ones every tutorial fills with the median. The join
operator `<=>` correlates two expressions (the first expression
drives, `$$0` names its row inside the join condition, `$$1` the
joined binding after); the outer form `<=>?` keeps unmatched rows
with a null witness, so an anti-join is a join plus a null
test.

In [6]:
%%quarb
/titanic/row[!::Age] @| count

177

In [7]:
%%quarb
/titanic-birth-records//*[::passenger_id <= 891] @| count

773

In [8]:
%%quarb
/titanic/row <=>? /titanic-birth-records//*[::passenger_id = $$0::PassengerId]
 | [!$$1::passenger_id] | ::Name @| [..6]

Vestrom, Miss. Hulda Amanda Adolfina
Hewlett, Mrs. (Mary D Kingcome) 
Vander Planke, Mrs. Julius (Emelia Maria Vandemoortele)
Todoroff, Mr. Lalio
Uruchurtu, Don. Manuel E
Vander Planke, Miss. Augusta Maria

In [9]:
%%quarb
/titanic-birth-records//*[::passenger_id > 891] @| count

373

773 of our rows have a record. The 118 without one are mostly
passengers born outside the eight covered systems — Belgians,
Bulgarians, a Mexican diplomat. And 373 records cite passengers
*beyond* id 891: the registers know the whole ship; the table only
ever held the training set.

## One year, eight spellings

A birth year is spelled `quarter: "1890 Q1"` in London,
`date: "3 October 1892"` in Mayo, `född: "1871-11-26"` in Sweden,
`year_rumi: 1301` in Ottoman Batroun, a Julian date in Kovno. One
conditional ladder normalizes all eight — each branch tests for a
key only one shape carries ("has this key" is how you say "is this
shape" when shapes have no names). Join the 177 age-less rows to
their records and read the answer:

In [10]:
%%quarb
def &gregorian: | tp("%Y-%m-%d") | ($_ + 12d);
/titanic/row[!::Age] <=> /titanic-birth-records//*[::passenger_id = $$0::PassengerId]
 | .born(( $$1::year_rumi      ? $$1::year_rumi + 584
         : $$1::quarter        ? ($$1::quarter | s/ Q.*$//) * 1
         : $$1::date           ? ($$1::date | s/^.*\s//) * 1
         : $$1::född           ? ($$1::född | s/-.*$//) * 1
         : $$1::né_le          ? ($$1::né_le | s/-.*$//) * 1
         : $$1::date_of_birth  ? ($$1::date_of_birth | s/-.*$//) * 1
         : $$1::natus          ? ($$1::natus | s/-.*$//) * 1
         : $$1::nata           ? ($$1::nata | s/-.*$//) * 1
         : (($$1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | %(::Name; born = $.born; age_in_1912 = 1912 - $.born) @| [..10]

{'Name': 'Moran, Mr. James', 'born': 1889, 'age_in_1912': 23}
{'Name': 'Williams, Mr. Charles Eugene', 'born': 1875, 'age_in_1912': 37}
{'Name': 'Masselmani, Mrs. Fatima', 'born': 1877, 'age_in_1912': 35}
{'Name': 'Emir, Mr. Farred Chehab', 'born': 1881, 'age_in_1912': 31}
{'Name': 'O\'Dwyer, Miss. Ellen "Nellie"', 'born': 1894, 'age_in_1912': 18}
{'Name': 'Spencer, Mrs. William Augustus (Marie Eugenie)', 'born': 1888, 'age_in_1912': 24}
{'Name': 'Glynn, Miss. Mary Agatha', 'born': 1895, 'age_in_1912': 17}
{'Name': 'Mamee, Mr. Hanna', 'born': 1885, 'age_in_1912': 27}
{'Name': 'Rogers, Mr. William John', 'born': 1871, 'age_in_1912': 41}
{'Name': 'Lennon, Mr. Denis', 'born': 1872, 'age_in_1912': 40}

In [11]:
%%quarb
/titanic/row[!::Age] <=> /titanic-birth-records//*[::passenger_id = $$0::PassengerId] @| count

157

**157 of the 177 missing ages were never missing** — they were
in the parish books, waiting for the data-engineering pass instead
of the median.

## The passengers who lied

For rows that *do* carry an age, compare it with the records
(people lied to shipping lines; registers did not). One join, one
pushed subtraction, one filter:

In [12]:
%%quarb
def &gregorian: | tp("%Y-%m-%d") | ($_ + 12d);
/titanic/row[::Age] <=> /titanic-birth-records//*[::passenger_id = $$0::PassengerId]
 | .born(( $$1::year_rumi      ? $$1::year_rumi + 584
         : $$1::quarter        ? ($$1::quarter | s/ Q.*$//) * 1
         : $$1::date           ? ($$1::date | s/^.*\s//) * 1
         : $$1::född           ? ($$1::född | s/-.*$//) * 1
         : $$1::né_le          ? ($$1::né_le | s/-.*$//) * 1
         : $$1::date_of_birth  ? ($$1::date_of_birth | s/-.*$//) * 1
         : $$1::natus          ? ($$1::natus | s/-.*$//) * 1
         : $$1::nata           ? ($$1::nata | s/-.*$//) * 1
         : (($$1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | .gap(::Age - (1912 - $.born))
 | [abs($.gap) >= 2]
 | %(::Name; claimed = ::Age * 1; records = 1912 - $.born)

{'Name': 'Heikkinen, Miss. Laina', 'claimed': 26, 'records': 24}
{'Name': 'Allen, Mr. William Henry', 'claimed': 35, 'records': 31}
{'Name': 'Saundercock, Mr. William Henry', 'claimed': 20, 'records': 15}
{'Name': 'Hickman, Mr. Stanley George', 'claimed': 21, 'records': 19}
{'Name': 'Hakkarainen, Mrs. Pekka Pietari (Elin Matilda Dolck)', 'claimed': 24, 'records': 22}
{'Name': 'Johnson, Miss. Eleanor Ileen', 'claimed': 1, 'records': 5}
{'Name': 'Youseff, Mr. Gerious', 'claimed': 45.5, 'records': 42}
{'Name': 'Givard, Mr. Hans Kristensen', 'claimed': 30, 'records': 25}
{'Name': 'Leyson, Mr. Robert William Norman', 'claimed': 24, 'records': 19}
{'Name': 'Maenpaa, Mr. Matti Alexanteri', 'claimed': 22, 'records': 17}
{'Name': 'Attalah, Mr. Sleiman', 'claimed': 30, 'records': 36}
{'Name': 'Stankovic, Mr. Ivan', 'claimed': 33, 'records': 27}
{'Name': 'Connolly, Miss. Kate', 'claimed': 22, 'records': 17}
{'Name': 'Bishop, Mrs. Dickinson H (Helen Walton)', 'claimed': 19, 'records': 25}
{'Name':

## The sisters nobody's ticket reveals

A birth record names *parents*, and two records that agree on both
parents are siblings whatever the manifest says. Self-join the
Swedish church books, then ask the manifest who each pair boarded
as — the final predicate sees all three bound contexts:

In [13]:
%%quarb
/titanic-birth-records//födda/*
 <=> /titanic/row[::PassengerId = $$0::passenger_id]
 <=> /titanic-birth-records//födda/*[::fader = $$0::fader
       && ::moder = $$0::moder && ::passenger_id > $$0::passenger_id]
 | [$$1::Name *= "Mrs."]
 | %(aboard_as = $$1::Name; sibling = $$2::namn)

{'aboard_as': 'Asplund, Mrs. Carl Oscar (Selma Augusta Emilia Johansson)', 'sibling': 'Gustaf Joel Johansson'}
{'aboard_as': 'Ahlin, Mrs. Johan (Johanna Persdotter Larsson)', 'sibling': 'Bengt Edvin Larsson'}
{'aboard_as': 'Backstrom, Mrs. Karl Alfred (Maria Mathilda Gustafsson)', 'sibling': 'Anders Vilhelm Gustafsson'}

Three married women, each traveling under a husband's name,
each with a brother aboard under the name she was born with. The
joining key — who your parents were — exists only below the
waterline.

## The names they left behind

Emigrants Westernized their names; the registers kept the
originals. Keep the records whose recorded surname does not appear
in the manifest name:

In [14]:
%%quarb
/titanic/row <=> /titanic-birth-records//*[::passenger_id = $$0::PassengerId]
 | [$$1::name] | .sur(($$1::name | s/^.*\s//))
 | [!(::Name *= $.sur)]
 | %(manifest = ::Name; record = $$1::name; village = $$1::village)

{'manifest': 'Novel, Mr. Mansouer', 'record': 'Mansouer Nawfal', 'village': 'Hardin'}
{'manifest': 'Peter, Miss. Anna', 'record': 'Anna Boutros', 'village': 'Shweir'}
{'manifest': 'Thomas, Master. Assad Alexander', 'record': 'Assad Tuma', 'village': 'Tula'}

In [15]:
%%quarb
/titanic/row <=> /titanic-birth-records//*[::passenger_id = $$0::PassengerId]
 | [$$1::familiya] | [!(::Name *= $$1::familiya)]
 | %(manifest = ::Name; record = "${$$1::imya} ${$$1::otchestvo} ${$$1::familiya}")

{'manifest': 'Abelson, Mr. Samuel', 'record': 'Shmuel Gershovich Abelzon'}
{'manifest': 'Paulner, Mr. Uscher', 'record': 'Usher Shmuelovich Polner'}

## The tip, at last — over to pandas

Now the data deserves a table: every passenger, the age the
manifest claims or the records recover, and the provenance of each
value. The same engine is an ordinary library, so the hand-off is
one call — `values()` returns plain dicts.

In [16]:
import pandas as pd
import quarb

root = quarb.mount(["titanic.csv", "titanic-birth-records.json"])
df = pd.DataFrame(root.values(r'''
def &gregorian: | tp("%Y-%m-%d") | ($_ + 12d);
/titanic/row <=>? /titanic-birth-records//*[::passenger_id = $$0::PassengerId]
 | .born(( $$1::year_rumi      ? $$1::year_rumi + 584
         : $$1::quarter        ? ($$1::quarter | s/ Q.*$//) * 1
         : $$1::date           ? ($$1::date | s/^.*\s//) * 1
         : $$1::född           ? ($$1::född | s/-.*$//) * 1
         : $$1::né_le          ? ($$1::né_le | s/-.*$//) * 1
         : $$1::date_of_birth  ? ($$1::date_of_birth | s/-.*$//) * 1
         : $$1::natus          ? ($$1::natus | s/-.*$//) * 1
         : $$1::nata           ? ($$1::nata | s/-.*$//) * 1
         : (($$1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | %(id = ::PassengerId * 1; ::Name; ::Sex;
     pclass = ::Pclass * 1; survived = ::Survived * 1;
     age = (( ::Age ? ::Age * 1 : 1912 - $.born ));
     age_source = (( ::Age ? "manifest" : $.born ? "records" : "unknown" )))
'''))
df.shape

(891, 7)

In [17]:
df["age_source"].value_counts()

age_source
manifest    714
records     157
unknown      20
Name: count, dtype: int64

In [18]:
df.groupby("age_source")["survived"].mean().round(3)

age_source
manifest    0.406
records     0.318
unknown     0.100
Name: survived, dtype: float64

From 177 unknown ages to 20 — and a finding the median would
have buried: the passengers whose ages had to be *recovered*
survived at 32%, against 41% for the rows the manifest filled in.
The missingness was never random — it tracks third class, and
imputing the median quietly erases that structure.

Pandas is now rightfully in charge. The rest of the iceberg —
eight archives, five joins, one conditional ladder — is what it
took to earn this table.